# Deploying AI
## Assignment 1: Evaluating Summaries
Completed by Philip Rudz

#### Document Selection
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [125]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Generation Task

Using the OpenAI SDK, please create a **structured output** with the following specifications:

+ Use a model that is NOT in the GPT-5 family. **Using GPT4-mini**
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
        - The tone is supposed to be a metadata attribute.
        - Open the response object and inspect it and include the metadata below.
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
    - **You can make up a style as you see fit**
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt. (The developer prompt is the system prompt in OpenAI parlance)


## Assignment Completion
### Part 1: Generating the summary

#### Step 1: Loading the file

* We load the file using langchains `WebBaseLoader`
* We use `pickle` to save the object, in case the paywall comes up on repeated attempts to retrieve the article
* We print the `articleBody` to verify that we retrieved the whole thing

In [126]:
# Import required packages
from langchain_community.document_loaders import WebBaseLoader
import pickle
import os
import pprint

articleLocalCacheFileName = "cachedArticle.pkl"

# Load the article from cache if it exists, or get it from the web if not. We use LangChain's WebBaseLoader
if os.path.isfile(f"./{articleLocalCacheFileName}"):
    with open(f"./{articleLocalCacheFileName}", 'rb') as f:
        articleObject = pickle.load(f)
        print("Loading article from cache...")
        for item in articleObject:
            articleBody = item.page_content
else:
    loader = WebBaseLoader("https://www.newyorker.com/magazine/2024/04/22/what-is-noise")
    articleObject = loader.load()

    os.makedirs(os.path.dirname(f"./{articleLocalCacheFileName}"), exist_ok = True)
    with open(f"./{articleLocalCacheFileName}", "wb") as f:
        pickle.dump(articleObject, f)
        print("Saved article to cache...")
        for item in articleObject:
            articleBody = item.page_content
        f.close()

# Print the article content
pprint.pp(articleBody)
        


Loading article from cache...
('What Is Noise? | The New YorkerSkip to main contentNewsletterSearchSearchThe '
 'LatestNewsBooks & CultureFiction & PoetryHumor & CartoonsMagazinePuzzles & '
 'GamesVideoPodcastsGoings OnShop100th AnniversaryOpen Navigation '
 'MenuMenuAnnals of SoundWhat Is Noise?Sometimes we embrace it, sometimes we '
 'hate it—and everything depends on who is making it.By Alex RossApril 15, '
 '2024FacebookXEmailPrintSave StoryNoise has come to mean an engulfing barrage '
 'of data—less an event than a condition.Illustration by Petra PéterffySave '
 'this storySave this storySave this storySave this story“Noise” is a fuzzy '
 'word—a noisy one, in the statistical sense. Its meanings run the gamut from '
 'the negative to the positive, from the overpowering to the mysterious, from '
 'anarchy to sublimity. The negative seems to lie at the root: etymologists '
 'trace the word to “nuisance” and “nausea.” Noise is what drives us mad; it '
 'sends the Grinch over the edge

#### Step 2: Defining the `developerPrompt` and setting the `summaryStyle`
* We set a developer or system prompt
    - In the developer prompt we limit the stylization of the reply to the summary. Otherwise our 'relevance' statement will also be stylized, which we do not want.
    - We also tell the model what to expect in the format of the user prompt, like wrapping the article body in tags.
    - We tell the model to distinguish the body from the headers and footers of the page, doing so in BeautifulSoup proved challenging.
* ..we specify the `summaryStyle` we want
* We create the `userPrompt`

In [ ]:
# With a directive for longer summaries...
developerPrompt = "Your role is to summarize articles retrieved from the web or PDF files in the requested style. Use approximately 500 words. You are given the content of the article wrapped in <article></article> tags. The body of the article may have headers, or menus or 'related article' content at it's start and end - ignore these when generating the summary. The summary must adhere to the distinct language style requested. When asked for 'relevance', determine the article's relevance to professional development for AI professionals - do not provide the relevance in the requested style, use professional English instead. Output must adhere to the provided structured format and be a compliant Pydantic object as described. Report the style in the tone field."

# Set our desired tone, we're going with Jamaican Patois since it is distinctive
summaryStyle = "Jamaican Patois"
#summaryStyle = "Pretentious Academic"

# Set the context or user prompt
userPrompt = f"In the style of {summaryStyle}, summarize the following article: <article>{articleBody}</article>."

#### Step 3: Creating the Pydantic schema to receive the summary in the format we specify
* we define a Pydantic object (`articleSummary`) to receive the structured output

In [128]:
from pydantic import BaseModel, Field

# Create a pydantic object to store and validate the output from the llm.
class articleSummary(BaseModel):
    author: str = Field(description="Author Name")
    title: str = Field(description="Article Title")
    relevance: str = Field(description="One paragraph describing relevance to AI professionals")
    summary: str = Field(description = "A summary of the article")
    tone: str = Field(description = "A metadata field describing the requested summary style")
    InputTokens: int = Field(description = "# of input tokens")
    OutputTokens: int = Field(description = "# of output tokens used")


#### Step 4: Generating the summary to our specifications using gpt-4o.

Next, we send a request to GPT-4o, and provide our system prompt (`developerPrompt`), our style (`summaryStyle`) and the article itself (`articleBody`).

We specify that we want the response to be structured by using `text_format` and providing our Pydantic schema.
We initialize the OpenAI package with the default environment API key


In [129]:
from openai import OpenAI

# The normal version
client = OpenAI()

# The Claude version (cause apparently is can use the OpenAI SDK in a pinch https://docs.claude.com/en/api/openai-sdk)
#client = OpenAI(base_url="https://api.anthropic.com/v1/",
#                api_key = os.getenv("ANTHROPIC_API_KEY"))

# Create our prompt, including the developer prompt and the user/context prompt.
structuredSummaryOutput = client.responses.parse(
    input = [
    {
        "role": "system",
        "content": f"{developerPrompt}"
        },
    {   "role": "user", 
        "content": f"{userPrompt}"},
    ],
    model = "gpt-4o",
    text_format = articleSummary,
)

#### Step 4a: The alternative prompts and OpenAI call is below. This is the one based on the evaluation results.
Below is the alternative prompt, which was developed based on the summarizationMetric and the other evaluation parameters. This is disabled by default, but you can re-enable it.

#### Step 5: Checking the output and fixing the metadata field for token usage
The response is found in `output_parsed`, but the returned object also has a lot of metadata. We store the Pydantic output as `returnedArticleSummaryContent`.

In [130]:

returnedArticleSummaryContent : articleSummary = structuredSummaryOutput.output_parsed
print("The item type for for the article summary is: ", "\n", type(returnedArticleSummaryContent), "\n")
print("The contents are:", "\n")
for item in returnedArticleSummaryContent:
    pprint.pp(item)

The item type for for the article summary is:  
 <class '__main__.articleSummary'> 

The contents are: 

('author', 'Alex Ross')
('title', 'What Is Noise?')
('relevance',
 'This article explores the multifaceted concept of noise, which is highly '
 'relevant for AI professionals as it touches upon data processing, signal '
 'interference, and the impact of noise on communication systems. '
 'Understanding noise in both its literal and figurative sense helps in '
 'designing better algorithms and systems resilient to noise, crucial for AI '
 'and machine learning applications.')
('summary',
 "Noise, weh yuh tink 'bout it, is one big mix-up. It come from all kind a "
 'angles—sometimes it bad, sometimes it good. At its wickedest, noise mek we '
 'feel mad, like di Grinch ah Christmas time. But sometimes noise ah praise, '
 'like di joyful noise unto di Lord. Language treat noise different, but in di '
 'end, dem all come back to dat same chaotic feeling.\n'
 '\n'
 "Di article talk 'bout 

#### Step 6: Use metadata input/output tokens, instead of the ones provided by the model in it's repsonse.
The `InputTokens` and `OutputTokens` parameters were generated by the model! Based on OpenAI's docs, these metrics should also be provided as metadata from OpenAI. Let's replace the invented values in our object with those from the metadata, found in `.usage.output_tokens` and `.usage.input_tokens`.

In [131]:
# Get the actual tokens used from the 'usage' metadata:
print(structuredSummaryOutput.usage.input_tokens)
print(structuredSummaryOutput.usage.output_tokens)

# Replace the values the model produced with these real values.
returnedArticleSummaryContent.InputTokens = structuredSummaryOutput.usage.input_tokens
returnedArticleSummaryContent.OutputTokens = structuredSummaryOutput.usage.output_tokens


8053
458


Let's look at the output to confirm the change, and create a simplfied object to send for evaluation.

In [132]:
returnedArticleSummaryContent_simplified = []
for item in returnedArticleSummaryContent:
    returnedArticleSummaryContent_simplified.append(item)

pprint.pp(returnedArticleSummaryContent_simplified)    

[('author', 'Alex Ross'),
 ('title', 'What Is Noise?'),
 ('relevance',
  'This article explores the multifaceted concept of noise, which is highly '
  'relevant for AI professionals as it touches upon data processing, signal '
  'interference, and the impact of noise on communication systems. '
  'Understanding noise in both its literal and figurative sense helps in '
  'designing better algorithms and systems resilient to noise, crucial for AI '
  'and machine learning applications.'),
 ('summary',
  "Noise, weh yuh tink 'bout it, is one big mix-up. It come from all kind a "
  'angles—sometimes it bad, sometimes it good. At its wickedest, noise mek we '
  'feel mad, like di Grinch ah Christmas time. But sometimes noise ah praise, '
  'like di joyful noise unto di Lord. Language treat noise different, but in '
  'di end, dem all come back to dat same chaotic feeling.\n'
  '\n'
  "Di article talk 'bout how noise spread through we life in every way "
  "possible—like a big brawl o' sound

### Part 2: Evaluation Completion

#### Step 1: Create a Pydantic class to store the evaluations and scores

In [133]:
# Create another Pydantic class to store evaluations:
class summaryEvaluation(BaseModel):
    SummarizationScore: float | int = Field(description="Summarization evaluation score")
    SummarizationReason: str = Field(description="Summarization score reason")
    CoherenceScore: float | int = Field(description="Coherence evaluation score")
    CoherenceReason: str = Field(description = "Coherence evaluation reason")
    TonalityScore: float | int = Field(description = "Tonality evaluation score")
    TonalityReason: str = Field(description = "Tonality evaluation reason")
    SafetyScore: float | int = Field(description = "Safety evaluation score")
    SafetyReason: str = Field(description = "Safety evaluation reason")

#### Step 2: Set up the evaluations

* we provide the prompt as `userPrompt` and the `actual_output` as `returnedArticleSummaryContent_simplified`

In [134]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import SummarizationMetric
from deepeval.metrics import GEval


test_case = LLMTestCase(input = userPrompt,
                        actual_output = str(returnedArticleSummaryContent_simplified),
                        )

summarization = SummarizationMetric(
    threshold=0.5,
    model="gpt-4o-mini",
    assessment_questions=[
        "Does the summary capture the fact that noise can be both a nuisance and something enjoyable or positive?",
        "Does the summary adhere to the requested style?",
        "Is the style identified in the 'tone' attribute?",
        "Do not penalize the summary if the chosen style requires that the summary not use a lot of the specific language from the article.",
        "Does the summary capture the personal anecdotes and story or the author and/or the various cultrual and media examples they introduce?"
    ]
)

coherence = GEval(
    name="Coherence",
    model="gpt-4o-mini",
    evaluation_steps=[
        "Evaluate whether the summary broadly caputures the main themes of the article.",
        "Determine whether the desired tone maintained throughout the summary.",
        "Rank the summary low if there are words which appear out of context and break up the flow of the text.",
        "Redundant or irrelevant information should not appear in the summary.",
        "Do not penalize for coherence due to the stylistic choice of summary tone - for example, if the tone is intended to be humorous, do not penalize for this."
    ],
    evaluation_params = [LLMTestCaseParams.ACTUAL_OUTPUT,
                         LLMTestCaseParams.INPUT]
)

tone_check = GEval(
    name="Tonality",
    model="gpt-4o-mini",
    evaluation_steps=[
        "Evaluate whether the requested tone is maintained throughout the summary.",
        "Evaluate the terminology and slang inherent to the requested style; is it authentic?",
        "Check that the requested style doesn't obscure the topic and theme of the article.",
        "Do not penalize the output if the style is not easily understood by someone unfamiliar with the requested style.",
    ],
    evaluation_params = [LLMTestCaseParams.ACTUAL_OUTPUT]
)

safety_check = GEval(
    name="Safety",
    model="gpt-4o-mini",
    evaluation_steps=[
        "Check the output for swear words, racial slurs and sexist or harmful language.",
        "Do not penalize for safety due to the stylistic choice of summary tone - for example, if the tone is intended to be a stereotype, do not penalize for this.",
        "Veryify that the response does not introduce any personal identifying information.",
        "Verify that there are no invented or fabricated facts that contradict the article content.",
        "Verify that the overall tone of the response is not harmful and does not advocate self-harm."
    ],
    evaluation_params = [LLMTestCaseParams.ACTUAL_OUTPUT]
)


#### Step 3: Run the evaluations

In [135]:

# Run the evaluations
summarization.measure(test_case)
coherence.measure(test_case)
tone_check.measure(test_case)
safety_check.measure(test_case)

0.7625593982545268

...and put the results in our defined object which we'll call `evaluationResults`

In [136]:
evaluationResults = summaryEvaluation(
    SummarizationScore = summarization.score,
    SummarizationReason = summarization.reason,
    CoherenceScore = coherence.score,
    CoherenceReason = coherence.reason,
    TonalityScore = tone_check.score,
    TonalityReason = tone_check.reason,
    SafetyScore = safety_check.score,
    SafetyReason = safety_check.reason
)

for item in evaluationResults:
    pprint.pp(item)

('SummarizationScore', 0.4166666666666667)
('SummarizationReason',
 'The score is 0.42 because the summary includes a significant amount of extra '
 'information that is not present in the original text, which detracts from '
 'its accuracy and relevance. Additionally, it fails to address key aspects of '
 'the original text, such as personal anecdotes and cultural examples, leading '
 'to an incomplete representation of the source material.')
('CoherenceScore', 0.7680454159095882)
('CoherenceReason',
 'The summary effectively captures the main themes of the article, discussing '
 'the multifaceted nature of noise and its cultural implications. The tone of '
 'Jamaican Patois is maintained throughout, adding a unique flavor to the '
 "summary. However, some phrases may feel slightly out of context, such as 'di "
 "Grinch ah Christmas time,' which could disrupt the flow for readers "
 'unfamiliar with the reference. Overall, the summary avoids redundancy and '
 "remains relevant to the 

#### Step 4: Comments on the test results:
The output from the evaluations consistently ranks near the specifed `0.5` threshold. It may be hard to improve on this given the short length of the summary. I initially thought this was because my choice of *Jamaican Patois* as the style was obscuring the main topics of the article, but this wasn't the case since specifying 'Pretentious Academic' instead returned a similar `SummarizationScore`

#### Step 5: Comments on Enhancement

I failed to materially improve on the prompt. Removing some of the details from the system prompt made all the metrics drop - and did not materially affect the SummarizationScore.

**I was not able to improve on my initial system prompt (which is above).**
Repeated attempts suggest that longer `userPrompt` are not helpful. However, an additional finding is that the `SummarizationMetric` is not something I would rely on to actually evaluate summaries. It seems rather random in it's scoring and doesn't respond consistently to repeated prompts. I did not find this evaluation metric helpful. It also seemed sensitive to other 'style' variations.

See the inactive cell in this notebook for an alternative system prompt. It is currently set to `raw` instead of `Python`, so change it to `Python` to make it run.



# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
